In [22]:
from pathlib import Path

import pandas as pd
import sqlalchemy as db
from tqdm import tqdm

In [10]:
def create_table(fp,
                 conn):
    import sqlalchemy as db
    from pathlib import Path

    with open(Path('../').joinpath(fp).absolute().resolve(), 'r') as f:
        text = f.read()
    text = text.replace('\t', ' ')
    text = text.replace('\n', '')
    statement = db.text(text)
    conn.execute(statement)
    conn.commit()

In [17]:
def add_to_server(d, engine):
    with engine.connect() as conn:
        files = [x for x in Path(d).iterdir()]
        for file in tqdm(files):
            df = pd.read_csv(file, index_col=0)
            df.to_sql('faces', conn, index=False, if_exists='append')

In [5]:
username = 'amos'
password = 'M0$hicat'
host = '192.168.0.131' 
port = '3306'
database = 'CineFace' 

In [6]:
connection_string = f'mysql+pymysql://{username}:{password}@{host}:{port}/{database}'
engine = db.create_engine(connection_string)

In [24]:
if not db.inspect(engine).has_table('faces'):
    with engine.connect() as conn:
        create_table('./sql/tables/faces.sql', conn)
        conn.commit()

In [25]:
add_to_server('../data/faces_pipeline/house', engine)